[<img src="https://colab.research.google.com/assets/colab-badge.svg" align="left" alt="Open In Colab"/>](https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/02_apigee_llm_gateway.ipynb)

<br><br>


# Apigee AI Gateway — LLM Governance & Policies (AFT)

**Repository Location:** [`https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/02_apigee_llm_gateway.ipynb`](https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/02_apigee_llm_gateway.ipynb)  
**Execution Constraint:** This notebook is version-controlled in GitHub and designed to run cleanly when opened directly from that repository path in Google Colab (no local-only paths, no machine-specific assumptions, and no state carried between sessions).

> **Scope Note:** **MCP tool-call validation** and **A2A agent-to-agent governance** are **OUT OF SCOPE** for this notebook — they are implemented in a separate notebook in the same repository. No structure or placeholders are reserved for them here. Summary and cleanup is numbered **Area 10**.

**Framework & Stability Rule:**
- **Toolkit:** **AFT (`aft` — Apigee Foundation Toolkit / Apigee Feature Templater)** (`https://github.com/apigee/apigee-templater`)
- **Base Repository:** [`https://github.com/gcp-samples/apigee-template-repository`](https://github.com/gcp-samples/apigee-template-repository)
- **Allowed Tiers Only:** Uses **ONLY** items published under **`## Stable Templates`** and **`## Stable Features`** in the base repository. Everything under **Drafts** (`## Draft Templates`, `## Draft Extension Features`, `features/draft/*`) is strictly excluded.
- **Assumptions:** Apigee X is already provisioned with `APIGEE_ENV` attached to an environment group, and target LLM backends are reachable.

---

## Step 0 — Preflight (Blocking Gate)
Run **Step 0 (`0.1`–`0.4`)** in order before running any later section.


In [ ]:
#@title 0.1 Inputs & Global Configuration
# Single input surface for Step 0: authenticates in Colab (same as 01_apigee_provisioning_global.ipynb),
# validates inputs, and exports environment variables.
import json, os, pathlib, re, shutil, subprocess, time, urllib.error, urllib.request
import requests

# Authenticate in Google Colab (runs once per Colab runtime session)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated successfully with Google Cloud in Colab.")
except ImportError:
    print("ℹ️ Running in local/standard Jupyter environment.")

#@markdown ### Project & Apigee
PROJECT_ID = "" #@param {type:"string"}
APIGEE_ORG = "" #@param {type:"string"}
APIGEE_ENV = "dev" #@param {type:"string"}
ENV_GROUP = "dev-group" #@param {type:"string"}
REGION = "europe-west1" #@param {type:"string"}
HOSTNAME = "" #@param {type:"string"}

#@markdown ### Service Account
SERVICE_ACCOUNT = "" #@param {type:"string"}

#@markdown ### Gemini
GEMINI_MODEL = "gemini-2.5-flash" #@param {type:"string"}

#@markdown ### Behaviour Flags
ENABLE_APIS = False #@param {type:"boolean"}

PROJECT_ID, APIGEE_ENV, ENV_GROUP, REGION, HOSTNAME, SERVICE_ACCOUNT = (
    v.strip() for v in (PROJECT_ID, APIGEE_ENV, ENV_GROUP, REGION, HOSTNAME, SERVICE_ACCOUNT)
)
APIGEE_ORG = APIGEE_ORG.strip() or PROJECT_ID
GEMINI_MODEL = GEMINI_MODEL.strip().removeprefix("google/")

missing = [k for k, v in {"PROJECT_ID": PROJECT_ID, "APIGEE_ENV": APIGEE_ENV, "REGION": REGION, "GEMINI_MODEL": GEMINI_MODEL}.items() if not v]
if missing:
    raise SystemExit(f"❌ HALT: Missing required parameter(s): {', '.join(missing)}.")

# If SERVICE_ACCOUNT is provided, do not ask for a name and do not create a new one
SA_PROVIDED = bool(SERVICE_ACCOUNT)
if SERVICE_ACCOUNT and "@" not in SERVICE_ACCOUNT:
    SERVICE_ACCOUNT = f"{SERVICE_ACCOUNT}@{PROJECT_ID}.iam.gserviceaccount.com"
if SERVICE_ACCOUNT and not re.fullmatch(r"[a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.gserviceaccount\.com", SERVICE_ACCOUNT):
    raise SystemExit(f"❌ HALT: Invalid SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' (expected *.gserviceaccount.com).")

os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID, "PROJECT_ID": PROJECT_ID, "APIGEE_ORG": APIGEE_ORG,
    "APIGEE_ENV": APIGEE_ENV, "ENV_GROUP": ENV_GROUP, "REGION": REGION, "HOSTNAME": HOSTNAME,
    "SERVICE_ACCOUNT": SERVICE_ACCOUNT, "GEMINI_MODEL": GEMINI_MODEL,
})
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID, "--quiet"], check=True, capture_output=True)

PREFLIGHT = {"inputs": True, "apigee": False, "service_account": False, "gemini": False, "aft": False}
sa_mode = f"{SERVICE_ACCOUNT} (provided — verify only, no creation)" if SA_PROVIDED else f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com (auto-provision if missing)"
print(f"✅ Inputs ready | project={PROJECT_ID} | org={APIGEE_ORG} | env={APIGEE_ENV} ({ENV_GROUP}) | region={REGION} | sa={sa_mode}")


In [ ]:
#@title 0.2 Verify Apigee Topology, APIs & IAM
# Uses the token from 0.1 via gcloud auth print-access-token (no authentication prompts here),
# verifies required APIs, resolves Apigee Org/Env/Hostname, and checks Apigee IAM permissions.
def get_auth_token() -> str:
    return subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()

def gcp_request(method: str, url: str, json_body=None, headers=None):
    req_headers = {"Authorization": f"Bearer {get_auth_token()}", "Content-Type": "application/json"}
    if headers:
        req_headers.update(headers)
    try:
        resp = requests.request(method, url, json=json_body, headers=req_headers, timeout=30)
        return resp.status_code, (resp.json() if resp.text else {})
    except ValueError:
        return resp.status_code, {"raw": resp.text}
    except Exception as exc:
        return 0, {"error": str(exc), "url": url}

ACTIVE_IDENTITY = subprocess.check_output(["gcloud", "config", "get-value", "account"], text=True).strip()
if not ACTIVE_IDENTITY:
    _, t_info = gcp_request("GET", f"https://oauth2.googleapis.com/tokeninfo?access_token={get_auth_token()}")
    ACTIVE_IDENTITY = t_info.get("email", "")
if not ACTIVE_IDENTITY:
    raise SystemExit("❌ HALT: Could not resolve active caller identity.")
caller_member_prefix = "serviceAccount" if ACTIVE_IDENTITY.endswith(".gserviceaccount.com") else "user"
print(f"✅ Active identity: {ACTIVE_IDENTITY}")

# 1. Verify required GCP APIs
REQUIRED_SERVICES = ["apigee.googleapis.com", "cloudresourcemanager.googleapis.com", "iamcredentials.googleapis.com", "aiplatform.googleapis.com"]
disabled_svcs = [
    svc for svc in REQUIRED_SERVICES
    if gcp_request("GET", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/{svc}")[1].get("state") != "ENABLED"
]
if disabled_svcs:
    if ENABLE_APIS or disabled_svcs == ["iamcredentials.googleapis.com"]:
        gcp_request("POST", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services:batchEnable", {"serviceIds": disabled_svcs})
        time.sleep(3)
    else:
        raise SystemExit(f"❌ HALT: Disabled API(s): {', '.join(disabled_svcs)}. Set ENABLE_APIS=True in 0.1 and re-run.")
print(f"✅ Required APIs ENABLED: {', '.join(REQUIRED_SERVICES)}")

# 2. Validate Apigee Organization, Environment, and Environment Group Hostname
APIGEE_BASE = f"https://apigee.googleapis.com/v1/organizations/{APIGEE_ORG}"
status, org_data = gcp_request("GET", APIGEE_BASE)
if status != 200 or org_data.get("state") != "ACTIVE":
    raise SystemExit(f"❌ HALT: Apigee org '{APIGEE_ORG}' not ACTIVE (HTTP {status}): {org_data}")

status, env_data = gcp_request("GET", f"{APIGEE_BASE}/environments/{APIGEE_ENV}")
if status != 200:
    raise SystemExit(f"❌ HALT: Apigee env '{APIGEE_ENV}' not found in org '{APIGEE_ORG}' (HTTP {status}): {env_data}")

_, eg_list = gcp_request("GET", f"{APIGEE_BASE}/envgroups")
matched_group, matched_hosts = None, []
for eg in eg_list.get("environmentGroups", []):
    eg_name = eg["name"]
    if ENV_GROUP and eg_name != ENV_GROUP:
        continue
    _, att_data = gcp_request("GET", f"{APIGEE_BASE}/envgroups/{eg_name}/attachments")
    if APIGEE_ENV in [a.get("environment") for a in att_data.get("environmentGroupAttachments", [])]:
        matched_group, matched_hosts = eg_name, eg.get("hostnames", [])
        break

if not matched_group or (not HOSTNAME and not matched_hosts):
    raise SystemExit(f"❌ HALT: Env '{APIGEE_ENV}' is not attached to a valid envgroup with a hostname in '{APIGEE_ORG}'.")
ENV_GROUP, HOSTNAME = matched_group, (HOSTNAME or matched_hosts[0])
os.environ.update({"ENV_GROUP": ENV_GROUP, "HOSTNAME": HOSTNAME})
print(f"✅ Apigee Topology: org={APIGEE_ORG} | env={APIGEE_ENV} | envgroup={ENV_GROUP} | host={HOSTNAME}")

# 3. Verify Effective Apigee IAM Permissions
REQUIRED_APIGEE_PERMS = [
    "apigee.proxies.create", "apigee.proxies.get", "apigee.proxies.update", "apigee.proxies.delete",
    "apigee.proxyrevisions.get", "apigee.proxyrevisions.update", "apigee.proxyrevisions.delete",
    "apigee.proxyrevisions.deploy", "apigee.proxyrevisions.undeploy",
    "apigee.deployments.create", "apigee.deployments.get", "apigee.deployments.update", "apigee.deployments.delete",
    "apigee.sharedflows.create", "apigee.sharedflows.get", "apigee.sharedflows.delete",
    "apigee.sharedflowrevisions.update", "apigee.sharedflowrevisions.deploy", "apigee.sharedflowrevisions.undeploy",
    "apigee.keyvaluemaps.create", "apigee.keyvaluemaps.get", "apigee.keyvaluemaps.update", "apigee.keyvaluemaps.delete",
    "apigee.keyvaluemapentries.create", "apigee.keyvaluemapentries.get", "apigee.keyvaluemapentries.update", "apigee.keyvaluemapentries.delete",
    "apigee.targetservers.create", "apigee.targetservers.get", "apigee.targetservers.update", "apigee.targetservers.delete",
]
iam_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:testIamPermissions"
_, iam_resp = gcp_request("POST", iam_url, {"permissions": REQUIRED_APIGEE_PERMS})
missing_perms = [p for p in REQUIRED_APIGEE_PERMS if p not in set(iam_resp.get("permissions", []))]
if missing_perms:
    raise SystemExit(f"❌ HALT: Missing Apigee IAM permissions: {', '.join(missing_perms)}")
PREFLIGHT["apigee"] = True
print(f"✅ Verified all {len(REQUIRED_APIGEE_PERMS)} Apigee IAM permissions.")


In [ ]:
#@title 0.3 Verify Service Account Roles, Permissions & <GoogleAccessToken> Bindings + Live Gemini
# If SERVICE_ACCOUNT was provided in 0.1, never creates a new one — verifies it exists and checks/ensures:
#   1) Project roles to call Gemini from Apigee (roles/aiplatform.user) and create tokens (roles/iam.serviceAccountTokenCreator)
#   2) Service-account resource bindings so Apigee runtime can impersonate it for <GoogleAccessToken>
REQUIRED_SA_ROLES = ["roles/aiplatform.user", "roles/iam.serviceAccountTokenCreator"]

def ensure_policy_bindings(get_url: str, set_url: str, required_pairs: list, label: str):
    status, policy = gcp_request("POST", get_url, {})
    if status != 200:
        raise SystemExit(f"❌ HALT: Failed to read {label} IAM policy (HTTP {status}): {policy}")
    bindings = policy.setdefault("bindings", [])
    changed = False
    for member, role in required_pairs:
        entry = next((b for b in bindings if b.get("role") == role and not b.get("condition")), None)
        if entry and member in entry.get("members", []):
            print(f"  ✔ {label} verified: {role} -> {member}")
        else:
            print(f"  ⏳ Binding {label}: {role} -> {member}")
            if entry:
                entry.setdefault("members", []).append(member)
            else:
                bindings.append({"role": role, "members": [member]})
            changed = True
    if changed:
        s_code, s_resp = gcp_request("POST", set_url, {"policy": policy})
        if s_code != 200:
            raise SystemExit(f"❌ HALT: Failed to update {label} IAM policy (HTTP {s_code}): {s_resp}")
        print(f"  ✔ {label} bindings updated.")

# 1. Resolve Project Number & Apigee Service Agent
_, proj_data = gcp_request("GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{APIGEE_ORG}")
if not proj_data.get("projectNumber"):
    _, proj_data = gcp_request("GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}")
if not proj_data.get("projectNumber"):
    raise SystemExit(f"❌ HALT: Unable to resolve projectNumber for '{PROJECT_ID}': {proj_data}")

PROJECT_NUMBER = str(proj_data["projectNumber"])
APIGEE_SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-apigee.iam.gserviceaccount.com"

# 2. Verify Provided Service Account (or Auto-Provision Default Only When Not Provided)
target_sa = SERVICE_ACCOUNT or f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com"
sa_status, sa_data = gcp_request("GET", f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{target_sa}")

if SA_PROVIDED:
    if sa_status != 200 or sa_data.get("disabled", False):
        raise SystemExit(f"❌ HALT: Provided SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' is missing or disabled (HTTP {sa_status}).")
    SA_STATUS = "VERIFIED_EXISTING"
    print(f"✅ Verified provided SERVICE_ACCOUNT: {SERVICE_ACCOUNT}")
elif sa_status == 200:
    SERVICE_ACCOUNT, SA_STATUS = target_sa, "REUSED_DEFAULT"
elif sa_status == 404:
    c_status, c_data = gcp_request(
        "POST",
        f"https://iam.googleapis.com/v1/projects/{PROJECT_ID}/serviceAccounts",
        {"accountId": "apigee-ai-gateway-sa", "serviceAccount": {"displayName": "Apigee AI Gateway Service Account"}},
    )
    if c_status not in (200, 201):
        raise SystemExit(f"❌ HALT: Failed to create default service account '{target_sa}' (HTTP {c_status}): {c_data}")
    SERVICE_ACCOUNT, SA_STATUS = c_data.get("email", target_sa), "CREATED_DEFAULT"
    time.sleep(3)
else:
    raise SystemExit(f"❌ HALT: Unable to inspect service account '{target_sa}' (HTTP {sa_status}): {sa_data}")

# 3. Verify & Ensure Project Roles (Gemini + Token Creator) and SA Resource Bindings (<GoogleAccessToken>)
proj_iam_base = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}"
ensure_policy_bindings(
    f"{proj_iam_base}:getIamPolicy",
    f"{proj_iam_base}:setIamPolicy",
    [(f"serviceAccount:{SERVICE_ACCOUNT}", r) for r in REQUIRED_SA_ROLES],
    "Project role",
)

sa_iam_base = f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{SERVICE_ACCOUNT}"
ensure_policy_bindings(
    f"{sa_iam_base}:getIamPolicy",
    f"{sa_iam_base}:setIamPolicy",
    [
        (f"serviceAccount:{APIGEE_SERVICE_AGENT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountUser"),
        (f"{caller_member_prefix}:{ACTIVE_IDENTITY}", "roles/iam.serviceAccountUser"),
    ],
    "SA <GoogleAccessToken> binding",
)

os.environ["SERVICE_ACCOUNT"] = SERVICE_ACCOUNT
PREFLIGHT["service_account"] = True
print(f"✅ Service Account ready for Gemini & <GoogleAccessToken>: {SERVICE_ACCOUNT} ({SA_STATUS})")

# 4. Verify Live Gemini Connectivity
payload = {"contents": [{"role": "user", "parts": [{"text": "Reply with READY."}]}], "generationConfig": {"maxOutputTokens": 8, "temperature": 0.0}}
def invoke_gemini(loc: str):
    host = "aiplatform.googleapis.com" if loc == "global" else f"{loc}-aiplatform.googleapis.com"
    url = f"https://{host}/v1/projects/{PROJECT_ID}/locations/{loc}/publishers/google/models/{GEMINI_MODEL}:generateContent"
    s, d = gcp_request("POST", url, payload, headers={"x-goog-user-project": PROJECT_ID})
    return gcp_request("POST", url, payload) if s == 403 else (s, d)

VERTEX_LOCATION, gen_status, gen_data = REGION, 0, {}
for _ in range(12):
    gen_status, gen_data = invoke_gemini(REGION)
    if gen_status == 200:
        VERTEX_LOCATION = REGION
        break
    for fb in [loc for loc in ("global", "us-central1") if loc != REGION]:
        fb_s, fb_d = invoke_gemini(fb)
        if fb_s == 200:
            gen_status, gen_data, VERTEX_LOCATION = fb_s, fb_d, fb
            break
    if gen_status == 200 or gen_status not in (403, 404):
        break
    time.sleep(5)

if gen_status != 200:
    raise SystemExit(f"❌ HALT: Live Gemini call failed (HTTP {gen_status}): {gen_data}")

os.environ["VERTEX_LOCATION"] = VERTEX_LOCATION
PREFLIGHT["gemini"] = True
print(f"✅ Live Gemini verified: model={GEMINI_MODEL} | location={VERTEX_LOCATION}")


In [ ]:
#@title 0.4 Install AFT CLI & Preflight Summary Gate
# Installs/verifies the AFT CLI, clones the GitHub repository for local AFT files, and confirms all Step 0 preflight gates passed.
# 1. Install / Verify AFT CLI
os.environ.pop("AFT_REPOSITORY", None)
for extra_bin in [str(pathlib.Path.home() / ".aft" / "bin"), str(pathlib.Path.home() / ".local" / "bin")]:
    if extra_bin not in os.environ["PATH"]:
        os.environ["PATH"] = f"{extra_bin}:{os.environ['PATH']}"

if not shutil.which("aft"):
    subprocess.run("curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh", shell=True, check=True)

aft_bin = shutil.which("aft")
if not aft_bin:
    raise SystemExit("❌ HALT: 'aft' binary not found on PATH after installation.")
aft_ver = subprocess.run([aft_bin, "--version"], capture_output=True, text=True, check=True).stdout.strip()
PREFLIGHT["aft"] = True
print(f"✅ AFT CLI ready: {aft_ver} ({aft_bin})")

# 2. Clone GitHub Repository (local templates & features for AFT)
PREFLIGHT["repo"] = False
REPO_URL = "https://github.com/luiscuellarh/apigee-unified-ai-gateway.git"
REPO_BRANCH = "main"
REPO_DIR = pathlib.Path.cwd() / "apigee-unified-ai-gateway"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True, capture_output=True, text=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True, capture_output=True, text=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True, capture_output=True, text=True)

if not (REPO_DIR / "templates").is_dir() or not (REPO_DIR / "features").is_dir():
    raise SystemExit(f"❌ HALT: Cloned repository at '{REPO_DIR}' is missing 'templates/' or 'features/' directories.")

WORKSPACE_DIR = REPO_DIR
os.environ.update({"REPO_URL": REPO_URL, "REPO_BRANCH": REPO_BRANCH, "REPO_DIR": str(REPO_DIR), "WORKSPACE_DIR": str(WORKSPACE_DIR)})
PREFLIGHT["repo"] = True
num_templates = len(list((REPO_DIR / "templates").glob("*.yaml")))
num_features = len(list((REPO_DIR / "features").glob("*.yaml")))
print(f"✅ Cloned GitHub repo ({REPO_BRANCH}): {REPO_DIR} ({num_templates} templates, {num_features} features)")

# 3. Preflight Summary Gate
failed_gates = [k for k, v in PREFLIGHT.items() if not v]
if failed_gates:
    raise SystemExit(f"❌ HALT: Preflight incomplete. Failed gate(s): {', '.join(failed_gates)}")

print("=" * 72)
for label, val in [
    ("GCP Project / No.",  f"{PROJECT_ID} ({PROJECT_NUMBER})"),
    ("Apigee Org / Env",   f"{APIGEE_ORG} / {APIGEE_ENV} ({ENV_GROUP})"),
    ("Gateway Hostname",   HOSTNAME),
    ("Active Identity",    ACTIVE_IDENTITY),
    ("Service Account",    f"{SERVICE_ACCOUNT} ({SA_STATUS}, <GoogleAccessToken> ready)"),
    ("Gemini Readiness",   f"PASS ({GEMINI_MODEL} @ {VERTEX_LOCATION})"),
    ("AFT Toolkit",        f"READY ({aft_ver})"),
    ("Local Repo (AFT)",   f"{REPO_DIR} (branch={REPO_BRANCH}, {num_templates} templates, {num_features} features)"),
]:
    print(f"{label:<20}: {val}")
print("=" * 72)
print("PREFLIGHT PASSED — safe to proceed")


## Area 1 — LLM Gateway (Completions)

### 1.1 Use Case Summary — LLM Gateway
- **What we are deploying:** An Apigee AI Gateway abstraction layer that accepts standard OpenAI-compatible `/v1/chat/completions` payloads and routes requests to Google Cloud Vertex AI.
- **Why it matters:** Client applications remain decoupled from underlying backend model endpoints and provider-specific wire formats, sharing one consistent API contract with project and routing configuration driven by Step 0 (`PROJECT_ID`, `REGION`, and `SERVICE_ACCOUNT`).
- **Template & Scope:** Built from [`templates/llm-gateway-completions-no-auth.yaml`](https://raw.githubusercontent.com/luiscuellarh/apigee-unified-ai-gateway/refs/heads/main/templates/llm-gateway-completions-no-auth.yaml). Consumer-facing authentication is deliberately out of scope in this `-no-auth` variant and is introduced in **Area 4**.


In [ ]:
#@title 1.2 Fetch Template, Map Parameters, Prep Workspace
# Fetches templates/llm-gateway-completions-no-auth.yaml (and its referenced ../features/*.yaml)
# from the authoring repository and maps Step 0 variables to the template parameter contract.
import json
import pathlib
import urllib.request
import urllib.error

AUTHORING_REPO_URL = "https://github.com/luiscuellarh/apigee-unified-ai-gateway"
RAW_REPO_BASE = "https://raw.githubusercontent.com/luiscuellarh/apigee-unified-ai-gateway/refs/heads/main"
TEMPLATE_REL_PATH = "templates/llm-gateway-completions-no-auth.yaml"
TEMPLATE_RAW_URL = f"{RAW_REPO_BASE}/{TEMPLATE_REL_PATH}"

REFERENCED_FEATURES = [
    "features/ai-endpoint-completions.yaml",
    "features/ai-target-googlecloud.yaml",
    "features/ai-model-preprocess.yaml",
    "features/ai-model-failover.yaml",
    "features/ai-model-postprocess.yaml",
    "features/ai-token-quota.yaml",
]

WORKSPACE_DIR = pathlib.Path.cwd() / "apigee-unified-ai-gateway-workspace"
(WORKSPACE_DIR / "templates").mkdir(parents=True, exist_ok=True)
(WORKSPACE_DIR / "features").mkdir(parents=True, exist_ok=True)

def fetch_repo_file(rel_path: str) -> pathlib.Path:
    url = f"{RAW_REPO_BASE}/{rel_path}"
    dest = WORKSPACE_DIR / rel_path
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "apigee-llm-gateway-colab"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            if resp.status != 200:
                raise SystemExit(f"❌ HALT: Failed to fetch '{url}' (HTTP {resp.status}).")
            content = resp.read().decode("utf-8")
    except urllib.error.URLError as exc:
        raise SystemExit(f"❌ HALT: Could not retrieve '{url}': {exc}")
    if not content.strip():
        raise SystemExit(f"❌ HALT: Retrieved empty file from '{url}'.")
    dest.write_text(content, encoding="utf-8")
    return dest

TEMPLATE_LOCAL_PATH = fetch_repo_file(TEMPLATE_REL_PATH)
for feat_rel in REFERENCED_FEATURES:
    fetch_repo_file(feat_rel)

os.environ["AFT_REPOSITORY"] = AUTHORING_REPO_URL

# Map Step 0 variables to the template parameter contract
AREA1_PARAMETERS = {
    "GoogleCloudProject": PROJECT_ID,
    "SimpleModel": "google/gemini-3.5-flash-lite",
    "MediumModel": "google/gemini-3.8-flash",
    "ComplexModel": "google/gemini-3.8-flash",
    "ThinkingModel": "anthropic/claude-opus-5",
    "ModelRouting": "'google/*'=googlecloud-oai,'zai-org/*=googlecloud-oai','anthropic/*'=googlecloud",
    "ModelMapping": f"gemini-flash-latest=gemini-3.7-flash,{GEMINI_MODEL}=gemini-3.7-flash",
}

print(f"✅ Template fetched: {TEMPLATE_RAW_URL}")
print(f"✅ Local path:       {TEMPLATE_LOCAL_PATH}")
print(f"✅ AFT_REPOSITORY:   {os.environ['AFT_REPOSITORY']}")
print("\n📋 Resolved Parameter Payload (Area 1 Contract):")
print(json.dumps(AREA1_PARAMETERS, indent=2))


In [ ]:
#@title 1.3 Render & Deploy via AFT
# Renders and deploys templates/llm-gateway-completions-no-auth.yaml to APIGEE_ENV using AFT.
import time

PROXY_NAME_LLM_ROUTING = os.environ.get("PROXY_NAME_LLM_ROUTING", "ai-model-routing-completions").strip() or "ai-model-routing-completions"
os.environ["PROXY_NAME_LLM_ROUTING"] = PROXY_NAME_LLM_ROUTING

# Pass scalar contract parameters via --parameters; comma-delimited routing/mapping strings
# are already defined in the fetched template defaults.
cli_param_str = ",".join(
    f"{k}={AREA1_PARAMETERS[k]}"
    for k in ("GoogleCloudProject", "SimpleModel", "MediumModel", "ComplexModel", "ThinkingModel")
)

aft_cmd = [
    aft_bin,
    "convert",
    "-i", str(TEMPLATE_LOCAL_PATH),
    "-n", PROXY_NAME_LLM_ROUTING,
    "--organization", APIGEE_ORG,
    "--environment", APIGEE_ENV,
    "--service-account", SERVICE_ACCOUNT,
    "--parameters", cli_param_str,
    "--token", get_auth_token(),
    "--no-animation",
]

print(f"⏳ Rendering and deploying proxy '{PROXY_NAME_LLM_ROUTING}' to env '{APIGEE_ENV}' via AFT...")
deploy_res = subprocess.run(aft_cmd, cwd=str(WORKSPACE_DIR), capture_output=True, text=True)
if deploy_res.returncode != 0:
    err_out = (deploy_res.stderr or deploy_res.stdout).strip()
    raise SystemExit(f"❌ HALT: AFT deployment failed (exit {deploy_res.returncode}):\n{err_out}")

# Poll Apigee environment deployments until the proxy revision reports READY
dep_url = f"{APIGEE_BASE}/environments/{APIGEE_ENV}/apis/{PROXY_NAME_LLM_ROUTING}/deployments"
deployed_rev, dep_state = None, "UNKNOWN"
for _ in range(24):
    d_status, d_data = gcp_request("GET", dep_url)
    deployments = d_data.get("deployments", []) if d_status == 200 else []
    if deployments:
        latest = sorted(deployments, key=lambda d: int(d.get("revision", 0)))[-1]
        deployed_rev = latest.get("revision")
        dep_state = latest.get("state", "READY")
        if dep_state == "READY":
            break
    time.sleep(5)

if dep_state != "READY":
    raise SystemExit(
        f"❌ HALT: Proxy '{PROXY_NAME_LLM_ROUTING}' did not reach READY state in env '{APIGEE_ENV}' "
        f"(revision={deployed_rev}, state={dep_state})."
    )

LLM_COMPLETIONS_ENDPOINT = f"https://{HOSTNAME}/v1/chat/completions"
os.environ["LLM_COMPLETIONS_ENDPOINT"] = LLM_COMPLETIONS_ENDPOINT

print(f"✅ Proxy Name:        {PROXY_NAME_LLM_ROUTING}")
print(f"✅ Deployed Revision: {deployed_rev} (state={dep_state})")
print(f"✅ Endpoint URL:      {LLM_COMPLETIONS_ENDPOINT}")


In [ ]:
#@title 1.4 Validation & Live Test
# Executes three live calls against the deployed /v1/chat/completions endpoint:
# 1) gemini-3.7-flash (expect HTTP 200)
# 2) gemini-3.1-pro   (expect HTTP 200 and model switch)
# 3) missing model    (expect HTTP 4xx rejection from OAS validation)
import time
import requests

def diagnose_gateway_failure(status_code: int, body) -> str:
    text = str(body).lower()
    if status_code in (401, 403) or "permission_denied" in text or "aiplatform.endpoints.predict" in text:
        return (
            f"Likely cause: Missing IAM permission or API disabled. Verify that service account "
            f"'{SERVICE_ACCOUNT}' holds 'roles/aiplatform.user' and 'aiplatform.googleapis.com' is ENABLED in '{PROJECT_ID}'."
        )
    if status_code == 404 or "not found" in text or "does not exist" in text:
        return (
            f"Likely cause: Requested model is not available on the Vertex AI endpoint in project '{PROJECT_ID}' "
            f"(region='{REGION}' / global) or the model ID is not published in Vertex AI Model Garden."
        )
    if status_code == 429 or "quota" in text or "resource_exhausted" in text:
        return f"Likely cause: Vertex AI quota or rate limit exhausted in project '{PROJECT_ID}'."
    return f"Likely cause: Upstream Vertex AI or gateway routing error (HTTP {status_code})."

def call_completions_endpoint(payload: dict):
    t0 = time.perf_counter()
    try:
        resp = requests.post(
            LLM_COMPLETIONS_ENDPOINT,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=45,
        )
    except requests.RequestException as exc:
        raise SystemExit(
            f"❌ HALT: Network/TLS error calling '{LLM_COMPLETIONS_ENDPOINT}': {exc}\n"
            f"Verify that load balancer IP/DNS and managed SSL certificate for '{HOSTNAME}' are ACTIVE."
        )
    latency_ms = (time.perf_counter() - t0) * 1000.0
    try:
        body = resp.json()
    except Exception:
        body = {"raw": resp.text}
    return resp.status_code, latency_ms, body

# Test 1: model = "gemini-3.7-flash"
status_1, latency_1, body_1 = call_completions_endpoint({
    "model": "gemini-3.7-flash",
    "messages": [{"role": "user", "content": "Reply with one short sentence confirming gemini-3.7-flash."}],
    "max_tokens": 32,
    "temperature": 0.0,
})
if status_1 != 200:
    print(f"❌ Test 1 (gemini-3.7-flash) returned HTTP {status_1}: {body_1}")
    print(f"🔎 {diagnose_gateway_failure(status_1, body_1)}")
    raise SystemExit("❌ HALT: Test 1 failed (expected HTTP 200).")

model_used_1 = body_1.get("model", "gemini-3.7-flash")
snippet_1 = (
    body_1.get("choices", [{}])[0]
    .get("message", {})
    .get("content", "")
    .strip()
    .replace("\n", " ")[:120]
)
print(f"✅ Test 1 PASS (HTTP {status_1}) | model={model_used_1} | latency={latency_1:.0f}ms | response={snippet_1!r}")

# Test 2: model = "gemini-3.1-pro"
status_2, latency_2, body_2 = call_completions_endpoint({
    "model": "gemini-3.1-pro",
    "messages": [{"role": "user", "content": "Reply with one short sentence confirming gemini-3.1-pro."}],
    "max_tokens": 32,
    "temperature": 0.0,
})
if status_2 != 200:
    print(f"❌ Test 2 (gemini-3.1-pro) returned HTTP {status_2}: {body_2}")
    print(f"🔎 {diagnose_gateway_failure(status_2, body_2)}")
    raise SystemExit("❌ HALT: Test 2 failed (expected HTTP 200).")

model_used_2 = body_2.get("model", "gemini-3.1-pro")
snippet_2 = (
    body_2.get("choices", [{}])[0]
    .get("message", {})
    .get("content", "")
    .strip()
    .replace("\n", " ")[:120]
)
if model_used_2 == model_used_1:
    raise SystemExit(f"❌ HALT: Expected model to switch from '{model_used_1}', but response still reported '{model_used_2}'.")
print(f"✅ Test 2 PASS (HTTP {status_2}) | model={model_used_2} (switched from {model_used_1}) | latency={latency_2:.0f}ms | response={snippet_2!r}")

# Test 3: Omit "model" — assert clean 4xx rejection
status_3, latency_3, body_3 = call_completions_endpoint({
    "messages": [{"role": "user", "content": "This request omits the required model field."}],
})
if not (400 <= status_3 < 500):
    raise SystemExit(f"❌ HALT: Test 3 (missing model) expected HTTP 4xx rejection, but got HTTP {status_3}: {body_3}")
print(f"✅ Test 3 PASS (HTTP {status_3}) | Missing 'model' cleanly rejected as expected ({latency_3:.0f}ms)")


## Area 2 — Prompt-Injection and Jailbreak Guardrails
- **Purpose:** Apply Google Cloud Model Armor screening to inspect incoming prompts and outgoing LLM completions for prompt-injection, jailbreak, and safety violations before returning responses.
- **Stable Template / Feature Citations:**
  - **Stable Features (`## Stable Features`):** `features/ai-security-modelarmor.yaml`
  - **Base Stable Templates (`## Stable Templates`):** `templates/REST-AI-GenerateContent.yaml`, `templates/REST-AI-Completions.yaml`
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing)


In [ ]:
#@title Area 2 — Prompt-Injection and Jailbreak Guardrails (Placeholder)
# TODO: Verify Model Armor API/template readiness, compose features/ai-security-modelarmor.yaml onto the AI proxy via aft, and test benign vs. prompt-injection payloads.
pass


## Area 3 — DLP / PII Detection and Redaction
- **Purpose:** De-identify and mask sensitive PII (such as emails, phone numbers, and credit card numbers) in user prompts before forwarding payloads to upstream LLM providers.
- **Stable Template / Feature Citations:**
  - **Stable Features (`## Stable Features`):** `features/security-anonymization-sdp.yaml`, `features/security-anonymization-presidio.yaml`
  - **Base Stable Templates (`## Stable Templates`):** `templates/REST-AI-GenerateContent.yaml`, `templates/REST-AI-Completions.yaml`
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing)


In [ ]:
#@title Area 3 — DLP / PII Detection and Redaction (Placeholder)
# TODO: Verify Cloud Sensitive Data Protection (dlp.googleapis.com) readiness, compose features/security-anonymization-sdp.yaml via aft, and test synthetic PII redaction.
pass


## Area 4 — Authentication and Authorization
- **Purpose:** Enforce client authentication and consumer group authorization on AI Gateway endpoints using Apigee API keys, OAuth 2.1 tokens, and KVM-backed group lookups while stripping client credentials before target invocation.
- **Stable Template / Feature Citations:**
  - **Stable Features (`## Stable Features`):** `features/auth-apikey-verify.yaml`, `features/auth-group-lookup.yaml`, `features/auth-oauth-verify.yaml`, `features/auth-oauth-server.yaml`
  - **Base Stable Templates (`## Stable Templates`):** `templates/REST-AI-Completions.yaml`, `templates/REST-AI-GenerateContent.yaml`
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing)


In [ ]:
#@title Area 4 — Authentication and Authorization (Placeholder)
# TODO: Provision an Apigee API Product, Developer, and Developer App, apply features/auth-apikey-verify.yaml and features/auth-group-lookup.yaml via aft, and test authorized vs. unauthorized requests.
pass


## Area 5 — Token Quota, Rate Limiting, Spike Arrest
- **Purpose:** Enforce LLM token consumption quotas per API Product/App (`total_tokens` weighted quota) using the published Stable Feature, while noting that Spike Arrest and KVM request rate-limiting exist only in Drafts.
- **Stable Template / Feature Citations:**
  - **Token Quota (`## Stable Features`):** `features/ai-token-quota.yaml` (pre-integrated in `templates/REST-AI-Completions.yaml`)
  - **Spike Arrest & Dynamic KVM Request Rate Limiting:** **`[NO STABLE OPTION]`** — `features/draft/spike-arrest-burst-smoother.yaml` and `features/draft/rate-limit-dynamic-kvm.yaml` sit under `## Draft Extension Features` and are excluded; only `features/ai-token-quota.yaml` is in Stable.
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing), Area 4 (Authentication and Authorization)


In [ ]:
#@title Area 5 — Token Quota, Rate Limiting, Spike Arrest (Placeholder)
# TODO: Configure token_quota, token_quota_interval, and token_quota_timeunit on the Apigee API Product and verify HTTP 200 within budget vs. HTTP 429 on token quota exhaustion.
pass


## Area 6 — Token Counting and Cost Attribution
- **Purpose:** Extract input, output, reasoning, and cached token counts from both non-streaming and SSE streaming LLM responses and calculate per-request cost using the `PriceList` entry in the `AI-Config` KVM.
- **Stable Template / Feature Citations:**
  - **Stable Features (`## Stable Features`):** `features/ai-model-postprocess.yaml`, `features/ai-model-preprocess.yaml`
  - **Base Stable Templates (`## Stable Templates`):** `templates/REST-AI-GenerateContent.yaml`, `templates/REST-AI-Completions.yaml`
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing)


In [ ]:
#@title Area 6 — Token Counting and Cost Attribution (Placeholder)
# TODO: Provision Apigee Data Collectors (dc_ai_model, dc_ai_provider, dc_input_token_count, dc_output_token_count, dc_cached_token_count, dc_reasoning_token_count, dc_total_token_count, dc_total_token_cost), seed PriceList in AI-Config KVM, and verify token/cost calculation.
pass


## Area 7 — Caching `[NO STABLE OPTION]`
- **Purpose:** Document the stability assessment for LLM response and semantic caching against the base repository.
- **Stable Template / Feature Citations:**
  - **`[NO STABLE OPTION]`:** Neither `## Stable Templates` nor `## Stable Features` includes an HTTP response cache or semantic cache policy (`features/draft/response-cache-invalidation.yaml` sits under `## Draft Extension Features` and is excluded).
  - **Closest Stable Templates as Base Proxy (`## Stable Templates`):** `templates/REST-AI-GenerateContent.yaml`, `templates/REST-AI-Completions.yaml` (note: `features/ai-model-postprocess.yaml` tracks upstream provider `cached_token_count` billing metrics, not gateway-layer response caching).
- **Dependencies:** Step 0 (Preflight)


In [ ]:
#@title Area 7 — Caching [NO STABLE OPTION] (Placeholder)
# TODO: No-op or await explicit instruction on how to handle Area 7 given [NO STABLE OPTION] in gcp-samples/apigee-template-repository.
pass


## Area 8 — Model Allow-Listing and Failover
- **Purpose:** Enforce per-product model allow-lists (`allowed_models` with wildcard/prefix matching) and automatic target failover to a backup Vertex AI model (`FailoverModel`) on upstream faults.
- **Stable Template / Feature Citations:**
  - **Model Allow-Listing (`## Stable Features`):** `features/ai-token-quota.yaml` (`JS-ValidateRequest` and `AM-ModelNotAllowed`)
  - **Model Failover (`## Stable Features`):** `features/ai-model-failover.yaml`, `features/ai-model-preprocess.yaml`
  - **Base Stable Templates (`## Stable Templates`):** `templates/REST-AI-Completions.yaml`, `templates/REST-AI-GenerateContent.yaml`
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing), Area 4 (Authentication and Authorization)


In [ ]:
#@title Area 8 — Model Allow-Listing and Failover (Placeholder)
# TODO: Configure allowed_models on the API Product and FailoverModel in the AI-Config KVM, compose features/ai-model-failover.yaml via aft, and test HTTP 400 model rejection and automatic failover.
pass


## Area 9 — Logging and Observability
- **Purpose:** Capture AI token, cost, model, and provider telemetry into Apigee Analytics via `DC-TokenAnalytics` and provision custom analytics reports, while noting external log/trace sinks exist only in Drafts.
- **Stable Template / Feature Citations:**
  - **Apigee AI Analytics & Data Collectors (`## Stable Features`):** `features/ai-model-postprocess.yaml` (`DC-TokenAnalytics` and `JS-Analytics`)
  - **External Log & Trace Sinks:** **`[NO STABLE OPTION]`** — `features/draft/cloud-logging-sink.yaml`, `features/draft/opentelemetry-trace-exporter.yaml`, `features/draft/datadog-metrics-forwarder.yaml`, and `features/draft/splunk-hec-forwarder.yaml` sit under `## Draft Extension Features` and are excluded.
- **Dependencies:** Step 0 (Preflight), Area 1 (LLM Routing), Area 6 (Token Counting and Cost Attribution)


In [ ]:
#@title Area 9 — Logging and Observability (Placeholder)
# TODO: Provision the AI Custom Analytics Reports (AI Token Cost by App & Model, AI Model Latency by Provider, AI Token Count by Consumer Group) and query Apigee Analytics dimensions.
pass


## Area 10 — Summary + Optional Cleanup
- **Purpose:** Print an inventory of all Apigee AI Gateway resources created during the session and optionally tear them down behind an explicit `CONFIRM_DELETE = False` `#@param` guard.
- **Stable Template / Feature Citations:** Notebook lifecycle utility operating on resources deployed from `## Stable Templates` and `## Stable Features`.
- **Dependencies:** Step 0 (Preflight), Areas 1–9


In [ ]:
#@title Area 10 — Summary + Optional Cleanup (Placeholder)
CONFIRM_DELETE = False #@param {type:"boolean"}
# TODO: Print deployed proxy/product/app/KVM/collector inventory and, only when CONFIRM_DELETE is True, undeploy and delete session resources cleanly.
pass
